# 작업형 제1유형 요점정리
> 생성일: 2026-05-05 | 기반: references/Pandas/ch01~ch08

## 목차
- [A. 데이터 로드 및 기본 확인](#A)
- [B. 결측치 처리](#B)
- [C. 이상치 처리](#C)
- [D. 통계 및 집계](#D)
- [E. 문자열 처리](#E)
- [F. 날짜/시간 처리](#F)
- [G. 데이터 변환](#G)
- [핵심 함수 요약](#summary)
- [오답 보강 섹션](#review)

In [ ]:
import pandas as pd
import numpy as np

---
<a id='A'></a>
## A. 데이터 로드 및 기본 확인

### 출제 포인트
- 데이터를 불러온 뒤 구조(shape, dtypes)와 분포(describe, skew)를 빠르게 파악
- 특정 컬럼의 고유값 수(`nunique`)나 빈도(`value_counts`)를 묻는 문제 자주 출제
- 주의: `encoding='cp949'` 한글 파일 처리 잊지 말 것

In [ ]:
# 데이터 로드
df = pd.read_csv('data.csv', encoding='utf-8-sig')

# 기본 확인
print(df.shape)          # (행수, 열수)
print(df.dtypes)         # 컬럼별 타입
df.info()                # Non-Null Count + dtype 요약
display(df.describe())   # 수치형 기초 통계

# 컬럼별 고유값 수
print(df.nunique())

# 특정 컬럼 빈도
print(df['category'].value_counts())
print(df['category'].value_counts(normalize=True).round(3))  # 비율

# 왜도/첨도
print(df['value'].skew())   # 왜도 (|값| > 1 이면 치우침 심함)
print(df['value'].kurt())   # 첨도

### 빈출 문제 패턴
> "category 컬럼에서 가장 많이 등장하는 값과 그 비율을 구하시오."

→ 대응 코드:

In [ ]:
df = pd.DataFrame({'category': ['A','B','A','C','A','B']})

top_val = df['category'].value_counts().index[0]
top_ratio = df['category'].value_counts(normalize=True).iloc[0]
print(top_val)
print(round(top_ratio, 3))

---
<a id='B'></a>
## B. 결측치 처리

### 출제 포인트
- 결측치 수 확인 → 전략 선택(고정값 / 평균 / ffill / dropna) → 처리 후 재확인
- `subset` 파라미터로 특정 컬럼 기준 삭제
- 주의: `fillna` 결과는 반드시 재할당해야 반영됨

In [ ]:
# 결측치 탐색
print(df.isnull().sum())                    # 컬럼별 결측치 수
print(df.isnull().sum()[df.isnull().sum() > 0])  # 결측치 있는 컬럼만

# 처리 전략 선택
df['num'] = df['num'].fillna(df['num'].mean())      # 평균으로 채우기
df['cat'] = df['cat'].fillna('unknown')              # 고정값
df['ts']  = df['ts'].ffill()                         # 시계열: 앞값
df['ts']  = df['ts'].bfill()                         # 시계열: 뒷값

# 행 삭제
df = df.dropna(subset=['target'], ignore_index=True) # target 컬럼 결측 행 제거

### 빈출 문제 패턴
> "결측치를 해당 컬럼의 중앙값으로 대체한 뒤 전체 평균을 구하시오."

In [ ]:
df = pd.DataFrame({'val': [1.0, np.nan, 3.0, np.nan, 5.0]})

df['val'] = df['val'].fillna(df['val'].median())
print(round(df['val'].mean(), 2))

---
<a id='C'></a>
## C. 이상치 처리

### 출제 포인트
- IQR 방식이 가장 자주 출제 (Q1-1.5×IQR, Q3+1.5×IQR 기준)
- `clip`으로 범위 밖 값을 경계값으로 교체하는 방식도 자주 출제
- 이상치 제거 후 특정 통계량을 구하는 2단계 문제 주의

In [ ]:
# IQR 방식 이상치 탐지 및 제거
Q1 = df['value'].quantile(0.25)
Q3 = df['value'].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

# 방법 1: 이상치 행 제거
df_clean = df[(df['value'] >= lower) & (df['value'] <= upper)].reset_index(drop=True)

# 방법 2: 경계값으로 대체 (clip)
df['value'] = df['value'].clip(lower=lower, upper=upper)

### 빈출 문제 패턴
> "IQR 방식으로 이상치를 제거한 뒤, value 컬럼의 평균을 소수점 둘째 자리로 구하시오."

In [ ]:
df = pd.DataFrame({'value': [10, 12, 13, 14, 15, 100, 11, 9, 200, 13]})

Q1, Q3 = df['value'].quantile([0.25, 0.75])
IQR = Q3 - Q1
df_clean = df[(df['value'] >= Q1 - 1.5*IQR) & (df['value'] <= Q3 + 1.5*IQR)]
print(round(df_clean['value'].mean(), 2))

---
<a id='D'></a>
## D. 통계 및 집계

### 출제 포인트
- `groupby + 집계함수` 조합이 핵심 (N번째로 큰 값, 조건 필터링 후 집계)
- `nlargest` / `nsmallest`로 상위/하위 N개 추출
- `rank`로 순위 매기기, `corr`로 상관계수 구하기
- `agg`로 여러 통계를 한 번에: `agg({'col': ['mean','std']})`

In [ ]:
# 그룹별 집계
result = df.groupby('group')['value'].mean().round(2)
print(result)

# 여러 집계 동시 적용
result = df.groupby('group').agg(
    avg=('value', 'mean'),
    total=('value', 'sum'),
    cnt=('value', 'count')
).round(2)

# 상위/하위 N개
print(df['value'].nlargest(3))   # 상위 3개
print(df['value'].nsmallest(3))  # 하위 3개

# 순위
df['rank'] = df['value'].rank(ascending=False, method='min')

# 상관계수
print(df[['A','B']].corr())
print(round(df['A'].corr(df['B']), 3))

### 빈출 문제 패턴
> "group별 value 합계를 구하고, 그 중 2번째로 큰 값을 출력하시오."

In [ ]:
df = pd.DataFrame({
    'group': ['A','A','B','B','C','C'],
    'value': [10, 20, 30, 40, 5, 15]
})

group_sum = df.groupby('group')['value'].sum().sort_values(ascending=False)
print(group_sum.iloc[1])  # 2번째로 큰 값

---
<a id='E'></a>
## E. 문자열 처리

### 출제 포인트
- `.str` accessor 체인 조합 (strip → replace → contains)
- `str.split(expand=True)`으로 분리 후 새 컬럼 생성
- `str.len()`으로 문자열 길이, `str.startswith/endswith`로 접두/접미 필터

In [ ]:
s = df['text']

# 기본 문자열 조작
s.str.strip()                          # 양쪽 공백 제거
s.str.replace('구', '시', regex=False) # 문자 교체
s.str.upper() / s.str.lower()          # 대/소문자
s.str.len()                            # 문자열 길이

# 조건 필터링
df[s.str.contains('서울')]             # '서울' 포함하는 행
df[s.str.startswith('김')]             # '김'으로 시작
df[s.str.endswith('점')]               # '점'으로 끝남

# 분리
df[['연도','월','일']] = df['날짜'].str.split('-', expand=True)

### 빈출 문제 패턴
> "이름이 '김'으로 시작하는 행의 수를 구하시오."

In [ ]:
df = pd.DataFrame({'name': ['김민준', '이서연', '김지호', '박수아', '김태양']})

count = df['name'].str.startswith('김').sum()
print(count)

---
<a id='F'></a>
## F. 날짜/시간 처리

### 출제 포인트
- `pd.to_datetime` 변환 후 `.dt` accessor로 연/월/일 추출 → groupby 집계
- 날짜 차이 계산: `(date2 - date1).dt.days`
- 연월 기준 groupby: `df.groupby(df['date'].dt.to_period('M'))`

In [ ]:
# 날짜 타입 변환
df['date'] = pd.to_datetime(df['date'], format='%Y-%m-%d')

# 날짜 정보 추출
df['year']    = df['date'].dt.year
df['month']   = df['date'].dt.month
df['day']     = df['date'].dt.day
df['weekday'] = df['date'].dt.dayofweek  # 0=월요일
df['quarter'] = df['date'].dt.quarter

# 날짜 차이 (일 수)
df['days_diff'] = (df['end_date'] - df['start_date']).dt.days

# 연월별 그룹 집계
monthly = df.groupby(df['date'].dt.to_period('M'))['value'].sum()

### 빈출 문제 패턴
> "월별 매출 합계를 구하고, 가장 매출이 높은 월을 출력하시오."

In [ ]:
df = pd.DataFrame({
    'date':  ['2024-01-10', '2024-01-25', '2024-02-05', '2024-02-20', '2024-03-15'],
    'sales': [100, 200, 150, 300, 80]
})
df['date'] = pd.to_datetime(df['date'])
df['month'] = df['date'].dt.month

monthly = df.groupby('month')['sales'].sum()
print(monthly.idxmax())          # 가장 높은 월
print(monthly.max())             # 해당 월 합계

---
<a id='G'></a>
## G. 데이터 변환

### 출제 포인트
- `sort_values` + `iloc` 조합으로 N번째 값 추출
- `cut/qcut`으로 구간화 후 집계
- `map(딕셔너리)`로 범주형 인코딩 (시험에서 get_dummies보다 선호)
- `merge/concat`으로 두 데이터셋 합치기

In [ ]:
# 정렬 후 N번째 값
df_sorted = df.sort_values('value', ascending=False).reset_index(drop=True)
print(df_sorted['value'].iloc[2])  # 3번째로 큰 값

# 구간화
df['grade'] = pd.cut(df['score'],
                     bins=[0, 60, 80, 100],
                     labels=['C', 'B', 'A'])
df['quartile'] = pd.qcut(df['score'], q=4, labels=['Q1','Q2','Q3','Q4'])

# 범주 인코딩 (map)
gender_map = {'남': 0, '여': 1}
df['gender_enc'] = df['gender'].map(gender_map)

# apply로 복합 조건 컬럼 생성
df['label'] = df['score'].apply(lambda x: '합격' if x >= 60 else '불합격')

# loc으로 조건부 값 수정
df.loc[df['score'] < 0, 'score'] = 0

# 데이터 이어붙이기
df_all = pd.concat([df1, df2], axis=0, ignore_index=True)

# 키 기반 병합
df_merged = pd.merge(df_left, df_right, on='id', how='left')

# 중복 제거
df = df.drop_duplicates(subset=['id'], keep='first').reset_index(drop=True)

### 빈출 문제 패턴
> "score를 기준으로 상위 25% 구간(Q4)에 속하는 행의 평균 age를 구하시오."

In [ ]:
df = pd.DataFrame({
    'score': [55, 70, 82, 91, 63, 78, 95, 48],
    'age':   [25, 30, 22, 28, 35, 27, 24, 31]
})

df['quartile'] = pd.qcut(df['score'], q=4, labels=['Q1','Q2','Q3','Q4'])
result = df[df['quartile'] == 'Q4']['age'].mean()
print(round(result, 2))

---
<a id='summary'></a>
## 핵심 함수 요약

| 카테고리 | 함수/메서드 | 용도 | 자주 쓰는 파라미터 |
|---------|------------|------|------------------|
| A | `pd.read_csv()` | CSV 로드 | `encoding`, `usecols`, `nrows` |
| A | `df.info()` | 구조 요약 | — |
| A | `df.describe()` | 기초 통계 | `include='all'` |
| A | `df['col'].value_counts()` | 빈도 | `normalize=True` |
| A | `df.nunique()` | 고유값 수 | — |
| B | `df.isnull().sum()` | 결측치 수 | — |
| B | `df['col'].fillna(v)` | 결측치 채우기 | 평균/중앙값/고정값 |
| B | `df.ffill()` / `df.bfill()` | 시계열 결측치 | — |
| B | `df.dropna()` | 결측치 행 삭제 | `subset`, `ignore_index=True` |
| C | `df['col'].quantile(0.25)` | 사분위수 | — |
| C | `df['col'].clip(lower, upper)` | 이상치 대체 | — |
| D | `df.groupby('col')['v'].mean()` | 그룹 집계 | — |
| D | `.agg(이름=('col','func'))` | 다중 집계 | — |
| D | `df['col'].nlargest(n)` | 상위 N개 | — |
| D | `df['col'].rank()` | 순위 | `ascending`, `method='min'` |
| D | `df.pivot_table()` | 교차표 | `values`,`index`,`columns`,`aggfunc` |
| E | `df['col'].str.contains('패턴')` | 포함 여부 | — |
| E | `df['col'].str.replace(a, b)` | 문자 교체 | `regex=False` |
| E | `df['col'].str.split(sep, expand=True)` | 분리 | — |
| F | `pd.to_datetime(df['col'])` | 날짜 변환 | `format='%Y-%m-%d'` |
| F | `df['col'].dt.year/month/day` | 날짜 추출 | — |
| F | `(d2-d1).dt.days` | 날짜 차이 | — |
| G | `df.sort_values('col', ascending=False)` | 정렬 | `na_position='last'` |
| G | `pd.cut(s, bins, labels)` | 등간격 구간화 | — |
| G | `pd.qcut(s, q, labels)` | 등빈도 구간화 | — |
| G | `df['col'].map(dict)` | 범주 인코딩 | — |
| G | `pd.concat([df1,df2], axis=0)` | 세로 병합 | `ignore_index=True` |
| G | `pd.merge(df1, df2, on, how)` | 키 기반 JOIN | `how='left'/'inner'` |
| G | `df.drop_duplicates(subset)` | 중복 제거 | `keep='first'` |

---
<a id='review'></a>
## 오답 보강 섹션
> (초기 생성 시 비어 있음 — 오답 분석 후 자동 추가됨)